# CARLA Simulation Demo: NAVSIM TransFuser

SageMaker AI Pipeline で学習した TransFuser モデルを CARLA シミュレーター上で実行し、走行動画を録画するデモ。

3 台の RGB カメラと LiDAR のセンサーデータを NAVSIM の特徴量フォーマットに変換し、モデルに入力して将来軌跡を予測します。

**前提条件**:
- GPU Notebook インスタンス (ml.g4dn.2xlarge 以上)
- 学習済み TransFuser モデル (navsim-transfuser-pipeline.ipynb または run-pipeline.sh で学習)

**入力センサー**:
- RGB カメラ 3 台 (左 60°、正面、右 60°) → スティッチして [3, 256, 1024] に変換
- LiDAR 1 台 → BEV ヒストグラム [1, 256, 256] に変換
- EgoStatus (速度、加速度、走行コマンド) → [8] ベクトル


## 1. 学習済みモデルの取得

Pipeline で学習した TransFuser の `model.pth` を S3 からダウンロードします。
最新の Training Job の出力から自動取得します。

まだ学習を実行していない場合は、先に以下のいずれかでモデルを学習してください。

```bash
# 方法 1: Notebook で学習
# navsim-transfuser-pipeline.ipynb を開いて実行

# 方法 2: ターミナルから一括実行
./pipelines/scripts/run-pipeline.sh -c container-navsim-transfuser
```

In [ ]:
import os
import tarfile
import boto3

# SageMaker AI Notebook の Lmod 環境変数が子プロセスでシェルエラーを起こすため除去
clean_env = {k: v for k, v in os.environ.items()
             if not k.startswith("BASH_FUNC_")}
os.environ.clear()
os.environ.update(clean_env)

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
MODEL_DIR = os.path.realpath(os.path.join(os.getcwd(), "..", "demo-carla", "transfuser", "model"))
MODEL_PATH = os.path.join(MODEL_DIR, "model.pth")

if os.path.exists(MODEL_PATH):
    print(f"Model already exists: {os.path.relpath(MODEL_PATH)}")
else:
    sm = boto3.client("sagemaker", region_name=REGION)
    paginator = sm.get_paginator("list_training_jobs")
    found = None
    for page in paginator.paginate(
        SortBy="CreationTime", SortOrder="Descending", StatusEquals="Completed",
    ):
        for summary in page["TrainingJobSummaries"]:
            job = sm.describe_training_job(TrainingJobName=summary["TrainingJobName"])
            image = job.get("AlgorithmSpecification", {}).get("TrainingImage", "")
            if "container-navsim-transfuser" in image:
                found = job
                break
        if found:
            break

    if not found:
        raise RuntimeError("No completed TransFuser training job found. Run the pipeline first.")

    s3_uri = found["ModelArtifacts"]["S3ModelArtifacts"]
    print(f"Training Job: {found['TrainingJobName']}")
    print(f"Downloading from: {s3_uri}")

    s3 = boto3.client("s3", region_name=REGION)
    bucket, key = s3_uri.replace("s3://", "").split("/", 1)
    tar_path = "/tmp/model.tar.gz"
    try:
        s3.head_object(Bucket=bucket, Key=key)
    except Exception:
        raise RuntimeError(
            f"Model artifact not found on S3: {s3_uri}\n"
            f"Training job '{found['TrainingJobName']}'"
            f" may still be running, or the artifact was deleted."
        ) from None
    s3.download_file(bucket, key, tar_path)

    os.makedirs(MODEL_DIR, exist_ok=True)
    with tarfile.open(tar_path) as tar:
        tar.extractall(MODEL_DIR, filter="data")
    print(f"Model saved to: {MODEL_PATH}")


## 2. CARLA インストール (初回のみ)

In [ ]:
import os
import shutil

CARLA_VERSION = "0.9.16"
CARLA_DIR = os.path.expanduser("~/SageMaker/carla")
CARLA_TAR = os.path.expanduser(f"~/SageMaker/CARLA_{CARLA_VERSION}.tar.gz")
CARLA_BIN = os.path.join(CARLA_DIR, "CarlaUE4.sh")

if os.path.exists(CARLA_BIN):
    print(f"CARLA already installed at {CARLA_DIR}")
else:
    if os.path.exists(CARLA_DIR):
        print("Removing incomplete installation...")
        shutil.rmtree(CARLA_DIR)

    print("Downloading CARLA (this takes a few minutes)...")
    !wget -q https://carla-releases.b-cdn.net/Linux/CARLA_{CARLA_VERSION}.tar.gz -O {CARLA_TAR}
    !mkdir -p {CARLA_DIR} && tar xzf {CARLA_TAR} -C {CARLA_DIR}
    !rm -f {CARLA_TAR}
    print(f"CARLA installed to {CARLA_DIR}")

## 3. 依存パッケージのインストール (初回のみ)

In [ ]:
import shutil
import importlib

missing = []
if not shutil.which("vulkaninfo"):
    missing.append("vulkan")
if not shutil.which("ffmpeg"):
    missing.append("ffmpeg")
for pkg in ["carla", "cv2", "timm"]:
    if importlib.util.find_spec(pkg) is None:
        missing.append(pkg)

if not missing:
    print("All dependencies already installed.")
else:
    print(f"Installing: {missing}")
    if "vulkan" in missing:
        !sudo yum install -y -q vulkan-loader vulkan-tools mesa-vulkan-drivers 2>/dev/null || true
    if "ffmpeg" in missing:
        !conda install -y -q -c conda-forge ffmpeg 2>/dev/null || true
    if any(p in missing for p in ["carla", "cv2", "timm"]):
        !pip install -q --cache-dir ~/SageMaker/.pip-cache carla==0.9.16 opencv-python "numpy<2" timm torchvision

## 4. CARLA サーバー起動

In [ ]:
import subprocess
import time

!pkill -9 -f CarlaUE4 2>/dev/null; sleep 2

carla_proc = subprocess.Popen(
    [f"{CARLA_DIR}/CarlaUE4.sh", "-RenderOffScreen", "--world-port=2000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print(f"Starting CARLA server (PID: {carla_proc.pid})...")

import carla
for i in range(60):
    time.sleep(5)
    try:
        client = carla.Client("localhost", 2000)
        client.set_timeout(5.0)
        ver = client.get_server_version()
        print(f"CARLA server ready (version: {ver})")
        break
    except Exception:
        print(f"  Waiting... ({(i+1)*5}s)")
else:
    carla_proc.kill()
    raise RuntimeError("CARLA server failed to start within 5 minutes")

## 5. 設定

In [ ]:
# 学習済みモデルのパス (Cell 1 で設定済み。変更する場合はここで上書き)
# MODEL_PATH = "/path/to/model.pth"

# シミュレーション設定
TOWN = "Town04"
DURATION_SEC = 30
OUTPUT_VIDEO = "outputs/transfuser_demo.mp4"


## 6. シミュレーション実行

CARLA 上で TransFuser モデルによる走行シミュレーションを実行し、動画を録画します。
3 台のカメラと LiDAR からリアルタイムでセンサーデータを取得し、モデルに入力します。
60 秒のシミュレーション + 動画変換で、完了まで数分かかります。

In [ ]:
import time as _time
_start = _time.time()

demo_dir = os.path.join(os.getcwd(), "..", "demo-carla", "transfuser")

!cd {demo_dir} && python run.py --model {MODEL_PATH} \
    --town {TOWN} --duration {DURATION_SEC} --output {OUTPUT_VIDEO}

print(f"\n✅ 完了 (所要時間: {_time.time() - _start:.0f} 秒)")

## 7. 動画再生

In [ ]:
from IPython.display import Video

video_path = os.path.join(demo_dir, OUTPUT_VIDEO)
Video(video_path, embed=True, width=960)

## 8. クリーンアップ

CARLA サーバーを停止します。

In [ ]:
carla_proc.kill()
carla_proc.wait()
print("CARLA server stopped.")